In [8]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from supervisor_agent.state import AppState
from langchain.chat_models import init_chat_model
from query_enhancer_agent.config import *
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel
from typing import Literal

llm=init_chat_model(model=MODEL, model_provider=MODEL_PROVIDER)

def analyse_query(appstate:AppState):
    class OutputFormat(BaseModel):
        output: Literal['True', 'False']
    query=appstate['query']
    prompt=PromptTemplate.from_template(
        """given a query, return TRUE or FALSE based on wether the query need to be decomposed
        query: {query}"""
    )
    llm_structred_output=llm.with_structured_output(OutputFormat)
    chain=prompt | llm_structred_output
    response=chain.invoke({"query":query})
    if response.output.lower()=='true':
        return {"should_decompose": True}
    return {"should_decompose": False}

def query_decomosition(appstate:AppState):
    query=appstate['query']
    prompt=PromptTemplate.from_template(
        """givena  query break it down into sub query so that it is easier to understand, only give the decomposed query nothing else
        query: {query}"""
    )
    chain=prompt | llm
    response=chain.invoke({"query":query})
    return {"response":response.content}

In [10]:
state = {
    "query": """
Why does my Lambda function timeout during initialization,
how can I debug the issue, and how should I monitor it?
"""
}

analyse_query(state)

{'should_decompose': True}

In [5]:
state = {"query": ["what is langchain and hows it differnt from langgraph"]}
query_decomosition(state)

{'response': '- What is LangChain?  \n- What are the main features and capabilities of LangChain?  \n- What is LangGraph?  \n- What are the main features and capabilities of LangGraph?  \n- How does LangGraph differ from LangChain?  \n- What are the key advantages and disadvantages of each?'}